# SQL Practice Exercises

## Setup

Run this cell first. In Colab, it downloads the flight database into `data/flight.db`; when working locally with the bundled database, it uses that copy. Then work through the exercises one cell at a time.

In [ ]:
import sqlite3
from pathlib import Path
from urllib.request import urlretrieve

db_path = Path("data/flight.db")
if not db_path.exists():
    db_path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/olearydj/INSY3010/main/lectures/12a-sql-exercises/data/flight.db"
    urlretrieve(url, db_path)
    print("Downloaded flight.db")

conn = sqlite3.connect(db_path)
cur = conn.cursor()


def run(sql):
    """Run a SQL query and print the results."""
    cur.execute(sql)
    cols = [desc[0] for desc in cur.description]
    rows = cur.fetchall()
    widths = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0)) for i, c in enumerate(cols)]
    header = " | ".join(c.ljust(w) for c, w in zip(cols, widths))
    print(header)
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"\n({len(rows)} rows)")

## Schema

```text
airplane(RegistrationNumber*, ModelNumber, Capacity)
booking(FlightNumber**, EmailAddress**)
flight(FlightNumber*, Origin, Destination, DepartureDate, DepartureTime,
       ArrivalDate, ArrivalTime, RegistrationNumber**)
passenger(EmailAddress*, GivenNames, Surname)
```

## Exercises

### 1a. What are the columns of the airplane table?

In [ ]:
# Trick question!
# SQLite uses PRAGMA instead of SHOW COLUMNS
# You don't need to know this syntax
sql = "PRAGMA table_info(airplane)"
run(sql)

#### Solution

*Concepts: SHOW COLUMNS vs SELECT, metadata vs data*

Server based SQL would use SHOW columns of airplane.

### 1b. Display all rows from the airplane table.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""

run(sql)

#### Solution

In [ ]:
sql = "SELECT * FROM airplane"
run(sql)

### 1c. What is the difference between the two results?

#### Solution

SHOW is for metadata (data about the database, tables, etc.). SELECT is for data IN the tables.

### 2. Display the first 10 passengers, sorted by surname.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: SELECT \*, FROM, ORDER BY, LIMIT*

In [ ]:
sql = """
SELECT *
FROM passenger
ORDER BY Surname
LIMIT 10
"""
run(sql)

### 3. List the origin, destination, and departure date for flights departing from Atlanta.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: SELECT columns, FROM, WHERE*

In [ ]:
sql = """
SELECT Origin, Destination, DepartureDate
FROM flight
WHERE Origin = 'Atlanta'
"""
run(sql)

### 4. Find all flights arriving on '11/7/15'. Show the flight number, origin, destination, and arrival date.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: WHERE with =, date values*

In [ ]:
sql = """
SELECT FlightNumber, Origin, Destination, ArrivalDate
FROM flight
WHERE ArrivalDate = '11/7/15'
"""
run(sql)

### 5. Find flights where the destination is not Atlanta or the departure date is '11/8/15'.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: WHERE with OR, != (NOT)*

In [ ]:
sql = """
SELECT FlightNumber, Origin, Destination, DepartureDate
FROM flight
WHERE Destination != 'Atlanta' OR DepartureDate = '11/8/15'
"""
run(sql)

### 6. List all flights departing from Miami, Orlando, or Dallas on '11/5/15'.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: WHERE with IN (...), AND*

In [ ]:
sql = """
SELECT FlightNumber, Origin, Destination, DepartureDate
FROM flight
WHERE Origin IN ('Miami', 'Orlando', 'Dallas')
  AND DepartureDate = '11/5/15'
"""
run(sql)

### 7. Which passenger has a missing surname?

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: WHERE with IS NULL*

In [ ]:
sql = """
SELECT *
FROM passenger
WHERE Surname IS NULL
"""
run(sql)

### 8. What airplane models are in our fleet, and what capacities do they come in? Show each unique model-capacity combination, sorted by capacity largest first.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: SELECT DISTINCT, ORDER BY DESC*

In [ ]:
sql = """
SELECT DISTINCT ModelNumber, Capacity
FROM airplane
ORDER BY Capacity DESC
"""
run(sql)

### 9. List all flights sorted by destination alphabetically, then by departure time latest first within each destination.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: ORDER BY multi-column, DESC on one column*

In [ ]:
sql = """
SELECT FlightNumber, Destination, DepartureDate, DepartureTime
FROM flight
ORDER BY Destination, DepartureTime DESC
"""
run(sql)

**Time-format note:** The supplied database stores departure times as text without consistently padding the hour to two digits. The query above therefore sorts `9:15:00` ahead of `21:20:00` in descending order, even though 21:20 is later. Text sorting matches time-of-day order when every value uses the fixed-width format HH:MM:SS.

The correction below pads the time for sorting without changing the database or the displayed values. `||` joins strings, and `substr('0' || DepartureTime, -8)` takes the final eight characters after prepending a zero: `9:15:00` becomes `09:15:00`, while `21:20:00` stays unchanged.

In [ ]:
sql = """
SELECT FlightNumber, Destination, DepartureDate, DepartureTime
FROM flight
ORDER BY Destination,
         substr('0' || DepartureTime, -8) DESC
"""
run(sql)

### 10. If each seat generates \\$150 in revenue per flight, show each airplane's registration number and its maximum revenue. Label the calculated column `MaxRevenue`. Only include airplanes where the maximum revenue exceeds \\$30,000.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: arithmetic in SELECT and WHERE, AS alias*

In [ ]:
sql = """
SELECT RegistrationNumber, Capacity * 150 AS MaxRevenue
FROM airplane
WHERE Capacity * 150 > 30000
"""
run(sql)

### 11. In a single query, find the total number of flights, the earliest departure date, and the latest departure date.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: COUNT(\*), MIN, MAX - aggregate functions without GROUP BY*

**Date-format note:** DepartureDate is stored as text. MIN and MAX compare these strings, which happen to sort chronologically for the supplied dates, November 5–8, 2015. This is not a general solution for dates written as M/D/YY. For reliable chronological comparisons across other dates, use a consistent ISO format (YYYY-MM-DD) or convert the values before comparing them.

In [ ]:
sql = """
SELECT COUNT(*) AS NumFlights,
       MIN(DepartureDate) AS Earliest,
       MAX(DepartureDate) AS Latest
FROM flight
"""
run(sql)

### 12. What is the average capacity of airplanes with more than 160 seats?

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: AVG with WHERE filter (no grouping)*

In [ ]:
sql = """
SELECT AVG(Capacity) AS AvgCapacity
FROM airplane
WHERE Capacity > 160
"""
run(sql)

### 13. Which are the top 3 busiest departure cities? Show the origin and flight count, sorted by count descending.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: GROUP BY, COUNT(\*), ORDER BY aggregate, LIMIT*

**Tie note:** Atlanta has 15 departures and Birmingham has 9. Houston, Memphis, and Miami tie for third with 7 each. This query returns only one of those tied cities; any of the three is acceptable. Add `Origin` as a secondary sort key (`ORDER BY FlightCount DESC, Origin`) for a reproducible choice. That selects Houston; it does not include all cities tied for third.

In [ ]:
sql = """
SELECT Origin, COUNT(*) AS FlightCount
FROM flight
GROUP BY Origin
ORDER BY FlightCount DESC
LIMIT 3
"""
run(sql)

### 14. For each airplane model, what is the total seating capacity across all planes of that model? Show the largest total first.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: GROUP BY, SUM, ORDER BY aggregate*

In [ ]:
sql = """
SELECT ModelNumber, SUM(Capacity) AS TotalCapacity
FROM airplane
GROUP BY ModelNumber
ORDER BY TotalCapacity DESC
"""
run(sql)

### 15. Which destinations are served by more than 5 flights? Show the destination and flight count, labeled `FlightCount`.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: GROUP BY, HAVING, AS*

In [ ]:
sql = """
SELECT Destination, COUNT(*) AS FlightCount
FROM flight
GROUP BY Destination
HAVING COUNT(*) > 5
"""
run(sql)

### 16. List each flight number alongside the capacity of the airplane assigned to it.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: INNER JOIN ... ON, table.column prefix*

In [ ]:
sql = """
SELECT flight.FlightNumber, airplane.Capacity
FROM flight
INNER JOIN airplane
  ON flight.RegistrationNumber = airplane.RegistrationNumber
"""
run(sql)

### 17. Show the airplane model and flight number for each flight departing from Birmingham. Sort by model.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: INNER JOIN, WHERE, ORDER BY*

In [ ]:
sql = """
SELECT airplane.ModelNumber, flight.FlightNumber
FROM flight
INNER JOIN airplane
  ON flight.RegistrationNumber = airplane.RegistrationNumber
WHERE Origin = 'Birmingham'
ORDER BY airplane.ModelNumber
"""
run(sql)

### 18. For each airplane model, how many flights are assigned to planes of that model?

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: INNER JOIN, GROUP BY, COUNT(\*)*

In [ ]:
sql = """
SELECT airplane.ModelNumber, COUNT(*) AS NumFlights
FROM airplane
INNER JOIN flight
  ON airplane.RegistrationNumber = flight.RegistrationNumber
GROUP BY airplane.ModelNumber
"""
run(sql)

### 19. Which passengers have no bookings? Show their email and surname.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: LEFT JOIN, WHERE ... IS NULL*

In [ ]:
sql = """
SELECT passenger.EmailAddress, passenger.Surname
FROM passenger
LEFT JOIN booking
  ON passenger.EmailAddress = booking.EmailAddress
WHERE booking.FlightNumber IS NULL
"""
run(sql)

### 20. For each passenger who has at least one booking, how many flights have they booked? Label the count `NumFlights` and sort by number of flights descending, then by surname.

In [ ]:
sql = """
-- your sql here
SELECT
FROM
"""
run(sql)

#### Solution

*Concepts: INNER JOIN, GROUP BY, COUNT(\*), HAVING, ORDER BY, AS*

In [ ]:
sql = """
SELECT passenger.Surname, passenger.GivenNames,
       COUNT(*) AS NumFlights
FROM passenger
INNER JOIN booking
  ON passenger.EmailAddress = booking.EmailAddress
GROUP BY passenger.EmailAddress, passenger.Surname, passenger.GivenNames
HAVING COUNT(*) >= 1
ORDER BY NumFlights DESC, passenger.Surname
"""
run(sql)

## Cleanup

In [ ]:
conn.close()